# 04 · Evaluadores y resultados del baseline

**Objetivo:** Programar los tres evaluadores y evaluar() y dejar congelados los resultados del baseline antes de mejorar nada.

**Requisitos:** R09, R10, R11, R12 ([01](../docs/01_requisitos_y_contratos.md)) · **Guía:** [12](../docs/12_skill_evaluadores.md) · [13 §2–§3](../docs/13_skill_medicion_informe_presentacion.md)

**Entradas:** `golden/golden_propio.jsonl` · **Salidas:** `resultados/baseline/`

**Independiente:** se ejecuta solo, sin ejecutar antes otros notebooks: lee `data/` y lo guardado en `resultados/`. Con `EJECUTAR = False` no llama a la API.

In [9]:
# Arranque (igual en todos los notebooks): localiza la raíz del repo y hace importable src/
import sys, pathlib, json, importlib
RAIZ = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
from IPython.display import display

from agente10k import config, datos, evaluacion

golden = evaluacion.cargar_golden(config.GOLDEN / "golden_propio.jsonl")
huecos = evaluacion.cargar_golden(config.GOLDEN / "golden_huecos.jsonl")
config.RESULTADOS.mkdir(parents=True, exist_ok=True)

print(f"{len(golden)} preguntas · {len(huecos)} huecos")
print(f"modelo: {config.MODELO_ID} · temperatura: {config.TEMPERATURA}")
print("resultados en:", config.RESULTADOS)

20 preguntas · 6 huecos
modelo: openrouter:google/gemini-3.8-flash · temperatura: 0
resultados en: C:\dev\10k-agent\resultados


In [10]:
# Parámetros
EJECUTAR = False  # True: llama al modelo (gasta API y pide clave). False: reutiliza resultados/
if EJECUTAR:
    assert config.cargar_clave(), "Falta OPENROUTER_API_KEY: copia .env.example a .env y rellénala"

## 0. Andamiaje: normalización y acceso a la verdad

Dos piezas antes de los evaluadores.

`normalizacion.py` trae `normalizar`, `cobertura`, `cifra_ok` y `normalizar_ticker`,
copiadas de [16 §3.2–3.3] y [09 §4]. **Una sola definición** de cada una: el
middleware XBRL del notebook 06 usará exactamente la misma `cifra_ok` que el
evaluador (b), que es lo que exige D07.

`datos.py` gana tres accesos indexados: `valor_xbrl`, `texto_seccion` y
`texto_chunk`.

In [11]:
"""Normalización de texto y cifras. Copiado de docs/16 §3.2-3.3 y docs/09 §4.

UNA sola definición de normalizar() y cifra_ok() en todo el proyecto (D07):
el middleware XBRL de R05 y el evaluador (b) de R09 usan estas mismas.
"""
from __future__ import annotations

import math
import re
import unicodedata

_TRAD = str.maketrans({"’": "'", "‘": "'", "“": '"', "”": '"', "–": "-", "—": "-"})

ALIAS = {"GOOG": "GOOGL", "ALPHABET": "GOOGL", "GOOGLE": "GOOGL", "FACEBOOK": "META",
         "NVIDIA": "NVDA", "MICROSOFT": "MSFT", "APPLE": "AAPL", "AMAZON": "AMZN"}

# "billones" (10^12) queda fuera a propósito: es el falso amigo de "billion".
ESCALA = [("miles de millones", 1e9), ("mil millones", 1e9), ("billion", 1e9),
          ("millones", 1e6), ("million", 1e6), ("miles", 1e3), ("thousand", 1e3)]


def normalizar(t: str | None) -> str:
    """Minúsculas, comillas tipográficas unificadas y espacios colapsados."""
    t = unicodedata.normalize("NFKC", t or "").translate(_TRAD).lower()
    return re.sub(r"\s+", " ", t).strip()          # colapsa \t y \n de las tablas


def normalizar_ticker(t) -> str:
    """La misma que usan las herramientas: admite alias y mayúsculas sueltas."""
    t = str(t or "").strip().upper()
    return ALIAS.get(t, t)


def _tokens(t: str) -> list[str]:
    return re.findall(r"\w+|[^\w\s]", normalizar(t))   # la puntuación, token aparte


def cobertura(frag: str, fuente: str, n: int = 4) -> float:
    """Fracción de n-gramas contiguos de `frag` presentes en `fuente`."""
    f, s = _tokens(frag), _tokens(fuente)
    if len(f) < n:
        return float(bool(f) and " ".join(f) in " ".join(s))
    vistos = set(zip(*(s[i:] for i in range(n))))
    grams = list(zip(*(f[i:] for i in range(n))))
    return sum(g in vistos for g in grams) / len(grams)


def cifra_ok(cifra, unidad, verdad, unidad_xbrl) -> dict:
    """Tolerancia documentada: USD 0,5 % relativo; USD/shares 0,005 absoluto.

    Hueco = acierto si cifra is None. Los desajustes de escala se etiquetan
    aparte de los de valor, para poder contarlos en la presentación.
    """
    if verdad is None:
        return {"ok": cifra is None, "motivo": "hueco"}
    if cifra is None:
        return {"ok": False, "motivo": "sin_cifra"}
    v = float(cifra) * next((f for clave, f in ESCALA if clave in normalizar(unidad)), 1.0)
    tol = ({"rel_tol": 0.0, "abs_tol": 0.005} if unidad_xbrl == "USD/shares"   # BPA: al céntimo
           else {"rel_tol": 0.005, "abs_tol": 0.0})                            # USD: 0,5 %
    if math.isclose(v, verdad, **tol):
        return {"ok": True}
    esc = next((e for e in (-9, -6, -3, 3, 6, 9) if math.isclose(v * 10**e, verdad, **tol)), None)
    return {"ok": False, "motivo": f"error_escala 1e{esc}" if esc else "fuera_de_tolerancia"}

In [12]:
# --- Accesos indexados para los evaluadores (docs/09 §4, docs/10 §6) ---

import functools


@functools.lru_cache(maxsize=1)
def _indice_xbrl() -> dict[tuple[str, int, str], tuple[float, str]]:
    x = cargar_xbrl()
    return {(t, int(f), c): (float(v), u) for t, f, c, v, u
            in zip(x["ticker"], x["fiscal_year"], x["concept"], x["value"], x["unit"])}


def valor_xbrl(ticker: str, fiscal_year: int, concept: str) -> tuple[float, str] | None:
    """(valor, unidad) del parquet, o None si ese hecho no está reportado."""
    try:
        return _indice_xbrl().get((ticker, int(fiscal_year), concept))
    except (TypeError, ValueError):
        return None


@functools.lru_cache(maxsize=1)
def _indice_secciones() -> dict[tuple[str, int, str], str]:
    s = cargar_secciones()
    return {(t, int(f), i): tx for t, f, i, tx
            in zip(s["ticker"], s["fiscal_year"], s["item"], s["texto"])}


def texto_seccion(ticker: str, fiscal_year: int, item: str) -> str:
    """El texto íntegro de una sección, o cadena vacía si no existe."""
    try:
        return _indice_secciones().get((ticker, int(fiscal_year), item), "")
    except (TypeError, ValueError):
        return ""


@functools.lru_cache(maxsize=1)
def _indice_chunks() -> dict[str, str]:
    c = cargar_chunks()
    return dict(zip(c["chunk_id"], c["texto"]))


def texto_chunk(chunk_id: str | None) -> str:
    """El texto de un fragmento, o cadena vacía si el identificador no existe."""
    return _indice_chunks().get(chunk_id or "", "")

## 1. Evaluador (b): la cifra contra XBRL

**Tolerancia documentada (va al informe):** USD relativa 0,5 %; USD/shares
absoluta 0,005; hueco = acierto si `cifra is None`.

La verdad sale del **parquet**, no del texto de la herramienta: es verificación
de estado, no de proceso. En comparativas se comprueban las dos cifras cuando el
agente devuelve `cifra_base`.

`b_tol{0, 0.1, 0.5, 1}` permite el análisis de sensibilidad del informe sin
volver a llamar al modelo.

In [13]:
"""Los cuatro evaluadores (docs/12). Parte 1: la verdad. Parte 2: la cifra (b)."""
from __future__ import annotations

import json
import math

from agente10k.datos import texto_chunk, texto_seccion, valor_xbrl
from agente10k.normalizacion import ESCALA, cifra_ok, cobertura, normalizar, normalizar_ticker

FYS, ITEMS = (2024, 2025), ("1A", "7", "7A", "8")
POR_DEFECTO = {"numerica": [["get_xbrl_fact"]],
               "extractiva": [["search_filings", "read_section"]],
               "comparativa": [["get_xbrl_fact"], ["search_filings", "read_section"]]}


def cargar_preguntas(ruta) -> list[dict]:
    """JSONL tolerante: no exige validar(); ids ausentes o repetidos se desambiguan."""
    with open(ruta, encoding="utf-8") as f:
        preguntas = [json.loads(x) for x in f if x.strip()]
    vistos: dict[str, int] = {}
    for n, p in enumerate(preguntas, 1):
        pid = str(p.get("id") or f"linea-{n}")
        vistos[pid] = vistos.get(pid, 0) + 1
        p["id"] = pid if vistos[pid] == 1 else f"{pid}#{vistos[pid]}"
    return preguntas


def _fy(d: dict, campo: str = "fiscal_year") -> int | None:
    try:
        return int(d.get(campo))
    except (TypeError, ValueError):
        return None


def fy_base(p: dict) -> int | None:
    """Comparativas: fiscal_year_base o, si falta, el otro FY de {2024, 2025}."""
    if (b := _fy(p, "fiscal_year_base")) is not None:
        return b
    fy = _fy(p)
    return next((x for x in FYS if x != fy), None) if fy in FYS else None


def verdad(p: dict, fy: int | None = None, base: bool = False) -> tuple:
    """(valor, unidad XBRL): del parquet si hay concept_xbrl; si no, del golden reescalado."""
    ticker = normalizar_ticker(p.get("ticker"))
    concepto, fy = p.get("concept_xbrl"), fy or _fy(p)
    if concepto and fy and (v := valor_xbrl(ticker, fy, concepto)) is not None:
        return v
    esperada = p.get("cifra_esperada_base" if base else "cifra_esperada")
    if esperada is None:
        return None, None
    u = normalizar(p.get("unidad"))
    factor = next((f for clave, f in ESCALA if clave in u), 1.0)
    return float(esperada) * factor, ("USD/shares" if "share" in u or "acci" in u else "USD")


def sin_referencia(p: dict) -> bool:
    """Ciegas que llegaran sin respuestas: no se pueden puntuar."""
    return all(p.get(k) is None for k in
               ("cifra_esperada", "concept_xbrl", "ancla_texto", "respuesta_esperada", "hueco"))


def es_hueco(p: dict) -> bool:
    """hueco:true, numérica sin cifra_esperada, o concepto inexistente para ese ticker y FY."""
    if p.get("hueco") is True or (p.get("familia") == "numerica" and p.get("cifra_esperada") is None):
        return True
    c, fy = p.get("concept_xbrl"), _fy(p)
    return bool(c and fy and valor_xbrl(normalizar_ticker(p.get("ticker")), fy, c) is None)


# --------------------------------------------------------------- (b) la cifra
def cifra_ok_rel(cifra, unidad, v, unidad_xbrl, rel: float) -> bool:
    """Sensibilidad: cambia solo la relativa de USD; el BPA y los huecos, como cifra_ok."""
    if v is None or cifra is None or unidad_xbrl == "USD/shares":
        return cifra_ok(cifra, unidad, v, unidad_xbrl)["ok"]
    x = float(cifra) * next((f for clave, f in ESCALA if clave in normalizar(unidad)), 1.0)
    return math.isclose(x, v, rel_tol=rel, abs_tol=0.0)


def evaluar_cifra(resp: dict, p: dict) -> dict:
    """(b): la cifra frente a XBRL; en comparativas, también cifra_base si viene."""
    if es_hueco(p):
        sin = resp.get("cifra") is None
        return {"b": sin, "b_motivo": "hueco" if sin else "hueco_inventado"}
    v, u = verdad(p)
    if v is None:
        return {"b": None, "b_motivo": "sin_verdad"}          # extractiva: (b) no aplica
    pares = [(resp.get("cifra"), v, u)]
    if p.get("familia") == "comparativa" and resp.get("cifra_base") is not None:
        vb, ub = verdad(p, fy_base(p), base=True)
        if vb is not None:
            pares.append((resp["cifra_base"], vb, ub))
    res = [cifra_ok(c, resp.get("unidad"), x, ux) for c, x, ux in pares]
    out = {"b": all(r["ok"] for r in res),
           "b_motivo": "; ".join(r.get("motivo") or "ok" for r in res),
           "b_verdad": v, "b_unidad_xbrl": u, "b_con_base": len(pares) == 2,
           "b_unidad_ok": normalizar(resp.get("unidad")) == normalizar(u)}
    for t in (0, 0.1, 0.5, 1):                                 # sensibilidad sin reejecutar
        out[f"b_tol{t}"] = all(cifra_ok_rel(c, resp.get("unidad"), x, ux, t / 100)
                               for c, x, ux in pares)
    return out

In [14]:
from agente10k import evaluadores, normalizacion
importlib.reload(normalizacion); importlib.reload(evaluadores)

p_num = next(p for p in golden if p["familia"] == "numerica")
p_ext = next(p for p in golden if p["familia"] == "extractiva")
p_eps = next(p for p in golden if p.get("unidad") == "USD/shares")
exacta = float(p_num["cifra_esperada"])
eps = float(p_eps["cifra_esperada"])

casos = [
    ("exacta",                 {"cifra": exacta, "unidad": "USD"},                  p_num,     True),
    ("redondeada 0,3 %",       {"cifra": exacta * 1.003, "unidad": "USD"},          p_num,     True),
    ("desviada 2 %",           {"cifra": exacta * 1.02, "unidad": "USD"},           p_num,     False),
    ("en millones, declarado", {"cifra": exacta / 1e6, "unidad": "millones de USD"},p_num,     True),
    ("en millones, sin decir", {"cifra": exacta / 1e6, "unidad": "USD"},            p_num,     False),
    ("sin cifra",              {"cifra": None, "unidad": None},                     p_num,     False),
    ("extractiva: no aplica",  {"cifra": None, "unidad": None},                     p_ext,     None),
    ("BPA exacto",             {"cifra": eps, "unidad": "USD/shares"},              p_eps,     True),
    ("BPA a 0,03",             {"cifra": eps + 0.03, "unidad": "USD/shares"},       p_eps,     False),
    ("hueco: se abstiene",     {"cifra": None, "unidad": None},                     huecos[0], True),
    ("hueco: se lo inventa",   {"cifra": 12345.0, "unidad": "USD"},                 huecos[0], False),
]

for nombre, resp, p, esperado in casos:
    r = evaluadores.evaluar_cifra(resp, p)
    ok = r["b"] is esperado
    print(f"{'ok ' if ok else 'MAL'} {nombre:24s} b={str(r['b']):5s} motivo={r.get('b_motivo')}")
    assert ok, nombre
print("\nevaluar_cifra pasa los 11 casos.")

ok  exacta                   b=True  motivo=ok
ok  redondeada 0,3 %         b=True  motivo=ok
ok  desviada 2 %             b=False motivo=fuera_de_tolerancia
ok  en millones, declarado   b=True  motivo=ok
ok  en millones, sin decir   b=False motivo=error_escala 1e6
ok  sin cifra                b=False motivo=sin_cifra
ok  extractiva: no aplica    b=None  motivo=sin_verdad
ok  BPA exacto               b=True  motivo=ok
ok  BPA a 0,03               b=False motivo=fuera_de_tolerancia
ok  hueco: se abstiene       b=True  motivo=hueco
ok  hueco: se lo inventa     b=False motivo=hueco

evaluar_cifra pasa los 11 casos.


## 2. Evaluador (c): la trayectoria

Acertar por el camino equivocado es fallo. Una cifra correcta leída de la prosa
da (b) ✓ pero (c) ✗, y el acierto de la pregunta cae.

Cuatro reglas, de [12 §4]:

- **Todas** las herramientas esperadas tienen que aparecer (AND, no OR).
- `list_available` es **neutra**: no se exige salvo en preguntas sobre el universo.
- **No se deduplica**: las llamadas repetidas y las bloqueadas cuentan.
- Además se comprueban los **argumentos** de `get_xbrl_fact`: ticker, ejercicio
  (los dos en comparativas) y concepto. La tolerancia sola no distingue la caja
  de META FY2024 (43.889 M) de su I+D (43.873 M): distan un 0,04 %.

In [15]:
# ------------------------------------------------------- (c) la trayectoria
REALES = {"list_available", "get_xbrl_fact", "search_filings", "read_section"}
CORPUS = {"NVDA", "MSFT", "AAPL", "GOOGL", "META", "AMZN"}


def esperadas(p: dict) -> tuple[list[list[str]], bool]:
    """(requisitos, por_defecto). Cada requisito es [nombre, *alternativas] (D12)."""
    lista = lambda x: [x] if isinstance(x, str) else list(x or [])
    h, alt = p.get("herramienta_esperada"), p.get("herramienta_alternativa")
    alt = alt if isinstance(alt, dict) else {}
    if h:
        return [[e, *lista(alt.get(e))] for e in lista(h)], False
    return [list(r) for r in POR_DEFECTO.get(p.get("familia"), [])], True


def _requisitos(p: dict) -> tuple[list[list[str]], bool]:
    req, por_defecto = esperadas(p)
    if req != [["list_available"]]:
        req = [r for r in req if r[0] != "list_available"]     # neutra, salvo universo
    return req, por_defecto


def _args_xbrl_ok(tcs: list[dict], p: dict) -> bool | None:
    """Alguna get_xbrl_fact con ticker, FY (los dos en comparativas) y concepto correctos."""
    concepto, fy = p.get("concept_xbrl"), _fy(p)
    if not concepto or fy is None:
        return None
    ticker, hueco = normalizar_ticker(p.get("ticker")), es_hueco(p)
    fys = {fy, fy_base(p)} - {None} if p.get("familia") == "comparativa" else {fy}
    vistos = set()
    for tc in tcs:
        a = tc.get("args") or {}
        f, c = _fy(a), a.get("concept")
        if tc["name"] != "get_xbrl_fact" or normalizar_ticker(a.get("ticker")) != ticker or f is None:
            continue
        v = valor_xbrl(ticker, f, c)
        # El concepto exacto, o uno de valor idéntico (GOOGL FY2024 tiene los dos de revenue).
        if c == concepto or (not hueco and v is not None and v == valor_xbrl(ticker, f, concepto)):
            vistos.add(f)
    return fys <= vistos


def evaluar_trayectoria(tool_calls: list[dict], p: dict, resp: dict) -> dict:
    """(c) = todas las herramientas esperadas ∧ argumentos correctos. Sin deduplicar."""
    tcs = [tc for tc in tool_calls if tc.get("name") in REALES]
    usadas = {tc["name"] for tc in tcs}
    req, por_defecto = _requisitos(p)
    tp = sum(bool(set(r) & usadas) for r in req)
    oblig = ["get_xbrl_fact"] in req
    en_corpus = normalizar_ticker(p.get("ticker")) in CORPUS
    args_ok = _args_xbrl_ok(tcs, p) if (oblig or "get_xbrl_fact" in usadas) and en_corpus else None
    c = (tp == len(req) and args_ok is not False) if req else None
    texto = bool(usadas & {"search_filings", "read_section"})
    coherencia = {"xbrl": "get_xbrl_fact" in usadas, "texto": texto,
                  "ambas": "get_xbrl_fact" in usadas and texto,
                  "ninguna": True}.get(resp.get("fuente"))
    return {"c": c, "c_recall": tp / len(req) if req else None, "c_args_ok": args_ok,
            "c_por_defecto": por_defecto, "coherencia_fuente": coherencia,
            "n_tools": len(tcs), "c_usadas": sorted(usadas)}

In [16]:
importlib.reload(evaluadores)

p_comp = next(p for p in golden if p["familia"] == "comparativa")
print(f"{p_num['id']} espera {p_num['herramienta_esperada']}")
print(f"{p_comp['id']} espera {p_comp['herramienta_esperada']} "
      f"(FY {p_comp.get('fiscal_year_base')} y {p_comp['fiscal_year']})\n")

def tc(*llamadas):
    """('get_xbrl_fact', {...}), ('search_filings', {}) -> lista de tool_calls."""
    return [{"name": n, "args": a, "id": f"c{i}"} for i, (n, a) in enumerate(llamadas)]

xb = lambda p, fy=None, con=None: ("get_xbrl_fact", {
    "ticker": p["ticker"], "fiscal_year": fy or p["fiscal_year"],
    "concept": con or p["concept_xbrl"]})

casos = [
    ("numérica: ruta y args ok",
     tc(xb(p_num)), p_num, {"fuente": "xbrl"}, True),
    ("numérica: concepto equivocado",
     tc(xb(p_num, con="StockholdersEquity")), p_num, {"fuente": "xbrl"}, False),
    ("numérica: ejercicio equivocado",
     tc(xb(p_num, fy=2024 if p_num["fiscal_year"] == 2025 else 2025)),
     p_num, {"fuente": "xbrl"}, False),
    ("numérica: camino equivocado",
     tc(("search_filings", {})), p_num, {"fuente": "texto"}, False),
    ("numérica: sin herramientas",
     tc(), p_num, {"fuente": "ninguna"}, False),
    ("comparativa: los dos años",
     tc(xb(p_comp), xb(p_comp, fy=p_comp["fiscal_year_base"]), ("search_filings", {})),
     p_comp, {"fuente": "ambas"}, True),
    ("comparativa: solo un año",
     tc(xb(p_comp), ("search_filings", {})), p_comp, {"fuente": "ambas"}, False),
    ("comparativa: falta el texto",
     tc(xb(p_comp), xb(p_comp, fy=p_comp["fiscal_year_base"])),
     p_comp, {"fuente": "xbrl"}, False),
]

for nombre, llamadas, p, resp, esperado in casos:
    r = evaluadores.evaluar_trayectoria(llamadas, p, resp)
    ok = r["c"] is esperado
    print(f"{'ok ' if ok else 'MAL'} {nombre:32s} c={str(r['c']):5s} "
          f"args={str(r['c_args_ok']):5s} usadas={r['c_usadas']}")
    assert ok, nombre
print("\nevaluar_trayectoria pasa los 8 casos.")

g3-001 espera ['get_xbrl_fact']
g3-014 espera ['get_xbrl_fact', 'search_filings'] (FY 2024 y 2025)

ok  numérica: ruta y args ok         c=True  args=True  usadas=['get_xbrl_fact']
ok  numérica: concepto equivocado    c=False args=False usadas=['get_xbrl_fact']
ok  numérica: ejercicio equivocado   c=False args=False usadas=['get_xbrl_fact']
ok  numérica: camino equivocado      c=False args=False usadas=['search_filings']
ok  numérica: sin herramientas       c=False args=False usadas=[]
ok  comparativa: los dos años        c=True  args=True  usadas=['get_xbrl_fact', 'search_filings']
ok  comparativa: solo un año         c=False args=False usadas=['get_xbrl_fact', 'search_filings']
ok  comparativa: falta el texto      c=False args=True  usadas=['get_xbrl_fact']

evaluar_trayectoria pasa los 8 casos.


## 3. Evaluador (a): la cita existe y respalda

Etapa 1, determinista: **existe** (el texto aparece literal en las secciones) ∧
**vista** (aparece en los `ToolMessage` de esa misma ejecución). La etapa 2,
`respalda`, la hace un juez y llega después.

"Vista" es la comprobación que caza al modelo que cita de memoria: una frase que
está en el corpus pero que el agente no recuperó no es evidencia, es recuerdo.

In [17]:
importlib.reload(evaluadores)

p_ext = next(p for p in golden if p["familia"] == "extractiva" and p.get("chunk_id_esperado"))
ancla = p_ext["ancla_texto"]
cid = p_ext["chunk_id_esperado"]
texto_visto = datos.texto_chunk(cid)

def obs(*textos):
    return [{"name": "search_filings", "tool_call_id": f"c{i}", "content": t}
            for i, t in enumerate(textos)]

base = {"ticker": p_ext["ticker"], "ejercicio": p_ext["fiscal_year"], "chunk_id": cid}

casos = [
    ("cita literal y vista",
     {**base, "cita": ancla}, obs(texto_visto), (True, True)),
    ("cita con elisión",
     {**base, "cita": f"{ancla[:40]} [...] {ancla[-40:]}"}, obs(texto_visto), (True, True)),
    ("existe pero NO la vio",
     {**base, "cita": ancla}, [], (True, False)),
    ("cita inventada",
     {**base, "cita": "The company will definitely double its revenue next year."},
     obs(texto_visto), (False, False)),
    ("sin cita",
     {**base, "cita": None}, obs(texto_visto), (False, False)),
]

for nombre, resp, observaciones, (exp_existe, exp_vista) in casos:
    r = evaluadores.etapa1_cita(resp, p_ext, observaciones)
    ok = (r["a_existe"], r["a_vista"]) == (exp_existe, exp_vista)
    print(f"{'ok ' if ok else 'MAL'} {nombre:24s} existe={str(r['a_existe']):5s} "
          f"vista={str(r['a_vista']):5s} chunk_ok={str(r.get('a_chunk_id_ok')):5s}"
          + (f" cobertura={r['a_cobertura']}" if "a_cobertura" in r else ""))
    assert ok, nombre
print("\netapa1_cita pasa los 5 casos.")

ok  cita literal y vista     existe=True  vista=True  chunk_ok=True 
ok  cita con elisión         existe=True  vista=True  chunk_ok=True 
ok  existe pero NO la vio    existe=True  vista=False chunk_ok=True 
ok  cita inventada           existe=False vista=False chunk_ok=False cobertura=0.0
ok  sin cita                 existe=False vista=False chunk_ok=None 

etapa1_cita pasa los 5 casos.


## 4. Abstención (d) y el acierto por familia

- numérica: (b) ∧ (c)
- extractiva: (a) ∧ (c) ∧ correcta
- comparativa: (a) ∧ (b) ∧ (c) ∧ correcta
- hueco: se abstiene ∧ sin cifra ∧ (c)

`abstencion_indebida` marca el caso que pidió el profesor: decir que no hay dato
cuando sí lo había.

Sin juez, las extractivas y comparativas dan `acierto = None`. La celda compara
el modo estricto con el provisional; la decisión de cuál usar va al informe.

In [18]:
importlib.reload(evaluadores)

def fila(p, resp, llamadas=(), observaciones=()):
    return {"id": p["id"], "golden": p, "respuesta": resp,
            "tool_calls": list(llamadas), "observaciones": list(observaciones)}

xbrl_ok = [{"name": "get_xbrl_fact", "id": "c0", "args": {
    "ticker": p_num["ticker"], "fiscal_year": p_num["fiscal_year"],
    "concept": p_num["concept_xbrl"]}}]

# numérica perfecta
s1 = evaluadores.puntuar_fila(fila(
    p_num, {"cifra": float(p_num["cifra_esperada"]), "unidad": "USD", "fuente": "xbrl"},
    xbrl_ok))
print(f"numérica ok            acierto={s1['acierto']}  b={s1['b']} c={s1['c']}")

# numérica correcta por el camino equivocado
s2 = evaluadores.puntuar_fila(fila(
    p_num, {"cifra": float(p_num["cifra_esperada"]), "unidad": "USD", "fuente": "texto"},
    [{"name": "search_filings", "id": "c0", "args": {}}]))
print(f"camino equivocado      acierto={s2['acierto']}  b={s2['b']} c={s2['c']}")

# hueco: se abstiene
h = huecos[0]
s3 = evaluadores.puntuar_fila(fila(
    h, {"cifra": None, "fuente": "ninguna"},
    [{"name": "get_xbrl_fact", "id": "c0", "args": {
        "ticker": h["ticker"], "fiscal_year": h["fiscal_year"], "concept": h["concept_xbrl"]}}]))
print(f"hueco: se abstiene     acierto={s3['acierto']}  abstiene={s3['abstiene']}")

# abstención indebida: dice "ninguna" en una pregunta que sí tiene dato
s4 = evaluadores.puntuar_fila(fila(
    p_num, {"cifra": None, "fuente": "ninguna"}, xbrl_ok))
print(f"abstención indebida    acierto={s4['acierto']}  indebida={s4['abstencion_indebida']}")

# extractiva sin juez: estricto frente a provisional
f_ext = fila(p_ext, {**base, "cita": ancla, "respuesta": "x", "fuente": "texto"},
             [{"name": "search_filings", "id": "c0", "args": {}}], obs(texto_visto))
print(f"extractiva estricta    acierto={evaluadores.puntuar_fila(f_ext)['acierto']}")
print(f"extractiva provisional acierto={evaluadores.puntuar_fila(f_ext, estricto=False)['acierto']}")

assert s1["acierto"] is True and s2["acierto"] is False
assert s3["acierto"] is True and s4["abstencion_indebida"] is True
print("\npuntuar_fila y acierto funcionan.")

numérica ok            acierto=True  b=True c=True
camino equivocado      acierto=False  b=True c=False
hueco: se abstiene     acierto=True  abstiene=True
abstención indebida    acierto=False  indebida=True
extractiva estricta    acierto=None
extractiva provisional acierto=True

puntuar_fila y acierto funcionan.
